In [8]:

import sys
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T


# ============================================================
# 1. AWS GLUE INITIALIZATION
# ============================================================

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)

job_name = "gold_healthcare_iot_job"

if "--JOB_NAME" in sys.argv:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job_name = args["JOB_NAME"]

job.init(job_name, {})

spark.conf.set("spark.sql.session.timeZone", "UTC")


# ============================================================
# 2. S3 CONFIGURATION
# ============================================================

SILVER = "s3://healthcare-iot-lake/silver"
GOLD = "s3://healthcare-iot-lake/gold"


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def read_silver(table_name):
    """Read a Silver table stored as Parquet."""
    path = f"{SILVER}/{table_name}/"
    print(f"Reading Silver table: {path}")
    return spark.read.parquet(path)


def write_gold(df, table_name, partition_by=None):
    """Replace the Gold output with the latest result."""
    path = f"{GOLD}/{table_name}/"

    writer = (
        df.write
        .mode("overwrite")
        .format("parquet")
    )

    if partition_by:
        writer = writer.partitionBy(*partition_by)

    writer.save(path)

    print(f"Gold table written: {path}")
    print(f"Rows written: {df.count()}")


def normalize_string(column_name):
    return F.upper(F.trim(F.col(column_name)))


def timestamp_column(column_name):
    # Handles ISO-8601 timestamps, including timestamps ending in Z.
    return F.to_timestamp(F.col(column_name))


# ============================================================
# 4. READ SILVER TABLES
# ============================================================

wards = read_silver("wards")
beds = read_silver("beds")
patients = read_silver("patients")
admissions = read_silver("admissions")
devices = read_silver("devices")
assignments = read_silver("device_assignments")
device_events = read_silver("device_status_events")
vitals = read_silver("vital_readings")


# ============================================================
# TABLE 1: GOLD WARD OCCUPANCY SUMMARY
# ============================================================
# Business purpose:
# Show occupied, available and maintenance beds per ward.
#
# Sources:
# Silver wards + beds
# ============================================================

bed_data = (
    beds.alias("b")
    .join(
        wards.alias("w"),
        F.col("b.ward_id") == F.col("w.ward_id"),
        "left"
    )
    .select(
        F.col("b.bed_id"),
        F.col("b.ward_id"),
        F.col("w.ward_name"),
        normalize_string("b.bed_status").alias("bed_status")
    )
)

ward_occupancy = (
    bed_data
    .groupBy("ward_id", "ward_name")
    .agg(
        F.count("bed_id").alias("total_beds"),

        F.sum(
            F.when(F.col("bed_status") == "OCCUPIED", 1)
             .otherwise(0)
        ).alias("occupied_beds"),

        F.sum(
            F.when(F.col("bed_status") == "AVAILABLE", 1)
             .otherwise(0)
        ).alias("available_beds"),

        F.sum(
            F.when(
                F.col("bed_status").isin(
                    "MAINTENANCE", "UNDER_MAINTENANCE"
                ),
                1
            ).otherwise(0)
        ).alias("maintenance_beds")
    )
    .withColumn(
        "occupancy_rate_pct",
        F.when(
            F.col("total_beds") > 0,
            F.round(
                F.col("occupied_beds") * 100.0 /
                F.col("total_beds"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn("snapshot_date", F.current_date())
)

write_gold(ward_occupancy, "ward_occupancy_summary")


# ============================================================
# TABLE 2: GOLD DEVICE RELIABILITY SUMMARY
# ============================================================
# Business purpose:
# Show which devices have many errors or disconnections.
#
# Sources:
# Silver devices + device_status_events
# ============================================================

normalized_events = (
    device_events
    .withColumn(
        "normalized_event_type",
        normalize_string("event_type")
    )
    .withColumn(
        "parsed_event_time",
        timestamp_column("event_time")
    )
)

event_counts = (
    normalized_events
    .groupBy("device_id")
    .agg(
        F.count("*").alias("total_events"),

        F.sum(
            F.when(
                F.col("normalized_event_type").isin(
                    "DISCONNECTED", "OFFLINE"
                ),
                1
            ).otherwise(0)
        ).alias("disconnect_count"),

        F.sum(
            F.when(
                F.col("normalized_event_type").isin(
                    "SENSOR_ERROR",
                    "ERROR",
                    "FAILURE"
                ),
                1
            ).otherwise(0)
        ).alias("error_count"),

        F.sum(
            F.when(
                F.col("normalized_event_type").isin(
                    "BATTERY_LOW", "CRITICAL_BATTERY"
                ),
                1
            ).otherwise(0)
        ).alias("battery_warning_count"),

        F.max("parsed_event_time").alias("last_event_at")
    )
)

device_reliability = (
    devices
    .select("device_id", "device_type", "model")
    .join(event_counts, "device_id", "left")
    .fillna({
        "total_events": 0,
        "disconnect_count": 0,
        "error_count": 0,
        "battery_warning_count": 0
    })
    .withColumn(
        "reliability_status",
        F.when(
            F.col("error_count") >= 3, "HIGH_ERROR_ACTIVITY"
        ).when(
            F.col("disconnect_count") >= 3,
            "FREQUENT_DISCONNECTIONS"
        ).when(
            F.col("battery_warning_count") >= 3,
            "FREQUENT_BATTERY_WARNINGS"
        ).when(
            F.col("total_events") == 0,
            "NO_EVENTS_RECORDED"
        ).otherwise("NO_REPEATED_INCIDENTS")
    )
    .withColumn("snapshot_date", F.current_date())
)

write_gold(device_reliability, "device_reliability_summary")


# ============================================================
# TABLE 3: GOLD ADMISSION SUMMARY
# ============================================================
# Business purpose:
# Show admissions by ward and admission status, with length
# of stay in days.
#
# Sources:
# Silver admissions + beds + wards
# ============================================================

admission_data = (
    admissions.alias("a")
    .join(
        beds.select("bed_id", "ward_id").alias("b"),
        F.col("a.bed_id") == F.col("b.bed_id"),
        "left"
    )
    .join(
        wards.select("ward_id", "ward_name").alias("w"),
        F.col("b.ward_id") == F.col("w.ward_id"),
        "left"
    )
    .select(
        F.col("a.admission_id"),
        F.col("a.patient_id"),
        F.col("b.ward_id"),
        F.col("w.ward_name"),
        normalize_string("a.status").alias("admission_status"),
        timestamp_column("a.admitted_at").alias("admitted_at"),
        timestamp_column("a.discharged_at").alias("discharged_at")
    )
    .withColumn(
        "length_of_stay_days",
        F.when(
            F.col("admitted_at").isNotNull(),
            F.round(
                (
                    F.coalesce(
                        F.col("discharged_at"),
                        F.current_timestamp()
                    ).cast("long")
                    - F.col("admitted_at").cast("long")
                ) / 86400.0,
                2
            )
        )
    )
)

admission_summary = (
    admission_data
    .groupBy(
        "ward_id",
        "ward_name",
        "admission_status"
    )
    .agg(
        F.countDistinct("admission_id").alias("admission_count"),
        F.countDistinct("patient_id").alias("distinct_patient_count"),
        F.round(
            F.avg("length_of_stay_days"), 2
        ).alias("average_length_of_stay_days"),
        F.round(
            F.max("length_of_stay_days"), 2
        ).alias("maximum_length_of_stay_days")
    )
    .withColumn("snapshot_date", F.current_date())
)

write_gold(admission_summary, "admission_summary")


# ============================================================
# TABLE 4: GOLD DEVICE INCIDENT SUMMARY
# ============================================================
# Business purpose:
# Summarize incidents by device, incident type and date.
#
# Sources:
# Silver device_status_events + devices
#
# These are operational device incidents, not patient
# medical diagnoses.
# ============================================================

incident_types = [
    "DISCONNECTED",
    "OFFLINE",
    "SENSOR_ERROR",
    "ERROR",
    "FAILURE",
    "BATTERY_LOW",
    "CRITICAL_BATTERY"
]

device_incidents = (
    normalized_events
    .filter(
        F.col("normalized_event_type").isin(*incident_types)
        & F.col("parsed_event_time").isNotNull()
    )
    .withColumn(
        "event_date",
        F.to_date("parsed_event_time")
    )
    .withColumn(
        "incident_severity",
        F.when(
            F.col("normalized_event_type").isin(
                "ERROR", "FAILURE", "SENSOR_ERROR",
                "CRITICAL_BATTERY"
            ),
            "HIGH"
        ).when(
            F.col("normalized_event_type").isin(
                "DISCONNECTED", "OFFLINE"
            ),
            "MEDIUM"
        ).otherwise("LOW")
    )
    .join(
        devices.select("device_id", "device_type", "model"),
        "device_id",
        "left"
    )
)

incident_summary = (
    device_incidents
    .groupBy(
        "device_id",
        "device_type",
        "model",
        "event_date",
        "normalized_event_type",
        "incident_severity"
    )
    .agg(
        F.countDistinct("event_id").alias("incident_count"),
        F.min("parsed_event_time").alias("first_incident_at"),
        F.max("parsed_event_time").alias("last_incident_at")
    )
)

write_gold(
    incident_summary,
    "device_incident_summary",
    partition_by=["event_date"]
)



# ============================================================
# 6. JOB COMPLETION
# ============================================================

print("All five Gold tables have been generated.")
job.commit()


Reading Silver table: s3://healthcare-iot-lake/silver/wards/
Reading Silver table: s3://healthcare-iot-lake/silver/beds/
Reading Silver table: s3://healthcare-iot-lake/silver/patients/
Reading Silver table: s3://healthcare-iot-lake/silver/admissions/
Reading Silver table: s3://healthcare-iot-lake/silver/devices/
Reading Silver table: s3://healthcare-iot-lake/silver/device_assignments/
Reading Silver table: s3://healthcare-iot-lake/silver/device_status_events/
Reading Silver table: s3://healthcare-iot-lake/silver/vital_readings/
Gold table written: s3://healthcare-iot-lake/gold/ward_occupancy_summary/
Rows written: 61
Gold table written: s3://healthcare-iot-lake/gold/device_reliability_summary/
Rows written: 975
Gold table written: s3://healthcare-iot-lake/gold/admission_summary/
Rows written: 116
Gold table written: s3://healthcare-iot-lake/gold/device_incident_summary/
Rows written: 350
All five Gold tables have been generated.


In [9]:

from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("PrintGoldTables").getOrCreate()

# S3 paths for the first 4 Gold tables
gold_tables = {
    "ward_occupancy_summary":
        "s3://healthcare-iot-lake/gold/ward_occupancy_summary/",

    "device_reliability_summary":
        "s3://healthcare-iot-lake/gold/device_reliability_summary/",

    "admission_summary":
        "s3://healthcare-iot-lake/gold/admission_summary/",

    "device_incident_summary":
        "s3://healthcare-iot-lake/gold/device_incident_summary/"
}

# Read and print each table
for table_name, s3_path in gold_tables.items():
    print("\n" + "=" * 80)
    print(f"TABLE: {table_name}")
    print("=" * 80)

    df = spark.read.parquet(s3_path)

    print(f"Total rows: {df.count()}")

    print("\nSchema:")
    df.printSchema()

    print("\nFirst 20 rows:")
    df.show(20, truncate=False)



TABLE: ward_occupancy_summary
Total rows: 61

Schema:
root
 |-- ward_id: string (nullable = true)
 |-- ward_name: string (nullable = true)
 |-- total_beds: long (nullable = true)
 |-- occupied_beds: long (nullable = true)
 |-- available_beds: long (nullable = true)
 |-- maintenance_beds: long (nullable = true)
 |-- occupancy_rate_pct: double (nullable = true)
 |-- snapshot_date: date (nullable = true)


First 20 rows:
+-------+----------------+----------+-------------+--------------+----------------+------------------+-------------+
|ward_id|ward_name       |total_beds|occupied_beds|available_beds|maintenance_beds|occupancy_rate_pct|snapshot_date|
+-------+----------------+----------+-------------+--------------+----------------+------------------+-------------+
|W24    |Pediatrics 4    |26        |18           |6             |2               |69.23             |2026-10-26   |
|W29    |Critical Care 5 |25        |17           |8             |0               |68.0              |2026-10

In [10]:

from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql import Row

# Initialize Spark for AWS Glue
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# Define the vital-sign detection rules
rules = [
    Row(
        rule_id="HR_LOW",
        metric_name="heart_rate",
        unit="BPM",
        rule_type="LOW",
        warning_operator="<",
        warning_threshold=50.0,
        critical_operator="<",
        critical_threshold=40.0,
        description="Heart rate is below the expected project threshold"
    ),
    Row(
        rule_id="HR_HIGH",
        metric_name="heart_rate",
        unit="BPM",
        rule_type="HIGH",
        warning_operator=">",
        warning_threshold=120.0,
        critical_operator=">",
        critical_threshold=150.0,
        description="Heart rate is above the expected project threshold"
    ),
    Row(
        rule_id="SPO2_LOW",
        metric_name="spo2",
        unit="%",
        rule_type="LOW",
        warning_operator="<",
        warning_threshold=90.0,
        critical_operator="<",
        critical_threshold=85.0,
        description="Blood oxygen saturation is low"
    ),
    Row(
        rule_id="SPO2_HIGH",
        metric_name="spo2",
        unit="%",
        rule_type="HIGH",
        warning_operator=">",
        warning_threshold=100.0,
        critical_operator=None,
        critical_threshold=None,
        description="Blood oxygen saturation exceeds 100 percent"
    ),
    Row(
        rule_id="TEMP_LOW",
        metric_name="temperature",
        unit="C",
        rule_type="LOW",
        warning_operator="<",
        warning_threshold=35.0,
        critical_operator="<",
        critical_threshold=32.0,
        description="Body temperature is below the project threshold"
    ),
    Row(
        rule_id="TEMP_HIGH",
        metric_name="temperature",
        unit="C",
        rule_type="HIGH",
        warning_operator=">",
        warning_threshold=38.5,
        critical_operator=">",
        critical_threshold=40.0,
        description="Body temperature is above the project threshold"
    ),
    Row(
        rule_id="SBP_LOW",
        metric_name="systolic_bp",
        unit="mmHg",
        rule_type="LOW",
        warning_operator="<",
        warning_threshold=90.0,
        critical_operator=None,
        critical_threshold=None,
        description="Systolic blood pressure is below the project threshold"
    ),
    Row(
        rule_id="SBP_HIGH",
        metric_name="systolic_bp",
        unit="mmHg",
        rule_type="HIGH",
        warning_operator=">",
        warning_threshold=180.0,
        critical_operator=None,
        critical_threshold=None,
        description="Systolic blood pressure is above the project threshold"
    )
]

# Convert rules into a Spark DataFrame
rules_df = spark.createDataFrame(rules)

# Save the Gold table in Parquet format
output_path = (
    "s3://healthcare-iot-lake/gold/vital_rules_summary/"
)

rules_df.write.mode("overwrite").parquet(output_path)

# Print the created table
print("Vital rules table created successfully.")
print(f"Output path: {output_path}")
print(f"Total rules: {rules_df.count()}")

rules_df.orderBy("metric_name", "rule_type").show(
    20, truncate=False
)


Vital rules table created successfully.
Output path: s3://healthcare-iot-lake/gold/vital_rules_summary/
Total rules: 8
+---------+-----------+----+---------+----------------+-----------------+-----------------+------------------+------------------------------------------------------+
|rule_id  |metric_name|unit|rule_type|warning_operator|warning_threshold|critical_operator|critical_threshold|description                                           |
+---------+-----------+----+---------+----------------+-----------------+-----------------+------------------+------------------------------------------------------+
|HR_HIGH  |heart_rate |BPM |HIGH     |>               |120.0            |>                |150.0             |Heart rate is above the expected project threshold    |
|HR_LOW   |heart_rate |BPM |LOW      |<               |50.0             |<                |40.0              |Heart rate is below the expected project threshold    |
|SPO2_HIGH|spo2       |%   |HIGH     |>            